In [1]:
import numpy as np
from sklearn.model_selection import train_test_split
from utils import train_categorical_model, prepare_datasets,validate_results,train_incident_agent,generate_oof_features
from rcf_model import RCF
from meta_scorer import train_fusion_meta_learner, CostSensitiveMetaLearner
from soar_zta import ZeroTrustSOARAgent

In [2]:
file_path = r"UNSW-NB15\unsw_train.csv"
X_train_cat, X_train_num, y_bin, cat_cols ,y_multi = prepare_datasets(file_path, is_train=True)

oof_cat_scores, oof_rcf_scores = generate_oof_features(
    X_train_cat, X_train_num, y_bin, cat_cols, 
    train_cat_func=train_categorical_model, 
    rcf_class=RCF, 
    n_splits=5
)

Loading and transforming data from UNSW-NB15\unsw_train.csv (Mode: Train)...
Final Categorical features: 6
Numerical features processed: 20 Principal Components

Generating 5-Fold OOF Predictions for Meta-Learner training...
  -> Processing Fold 1/5...

Training CatBoost (with High Regularization)...
Building RRCF Forest (40 trees)...


RRCF Blind Test Scoring: 100%|██████████| 16467/16467 [02:36<00:00, 104.97it/s]


  -> Processing Fold 2/5...

Training CatBoost (with High Regularization)...
Building RRCF Forest (40 trees)...


RRCF Blind Test Scoring: 100%|██████████| 16467/16467 [02:34<00:00, 106.32it/s]


  -> Processing Fold 3/5...

Training CatBoost (with High Regularization)...
Building RRCF Forest (40 trees)...


...

  -> Processing Fold 4/5...

Training CatBoost (with High Regularization)...
Building RRCF Forest (40 trees)...


RRCF Blind Test Scoring: 100%|██████████| 16466/16466 [02:26<00:00, 112.56it/s]


  -> Processing Fold 5/5...

Training CatBoost (with High Regularization)...
Building RRCF Forest (40 trees)...


RRCF Blind Test Scoring: 100%|██████████| 16466/16466 [02:13<00:00, 123.52it/s]

OOF Generation Complete.


In [3]:
X_meta_train_clean = np.column_stack((oof_cat_scores, oof_rcf_scores))

# Train the cost-sensitive meta-learner (it will internally scale and save its state)
meta_learner = train_fusion_meta_learner(
    X_meta_train_clean, 
    y_bin, 
    COST_FN=10, 
    COST_FP=2, 
    lambda_reg=0.1
)

meta_learner.save_model("Saves/meta_learner.pkl")


Training Meta-Learner (FN Penalty: 10, FP Penalty: 2, L2: 0.1)...
      ↳ Convergence reached at epoch 2082.
Meta-Learner Training Complete!
💾 Model state successfully saved to Saves/meta_learner.pkl


In [4]:
#CatBoost for Categorical Features
catboost_model = train_categorical_model(X_train_cat, y_bin, cat_cols)
incident_model = train_incident_agent(X_train_cat, y_multi, cat_cols)
cat_scores_raw = catboost_model.predict_proba(X_train_cat)[:, 1]

print("\nSaving Base Models to disk...")
catboost_model.save_model(r"Saves/catboost_base.cbm")
incident_model.save_model(r"Saves/incident_base.cbm")


Training CatBoost (with High Regularization)...

Training Incident Agent (Multi-class Threat Classifier)...

Saving Base Models to disk...


In [5]:
rcf_model = RCF(num_trees=40, tree_size=256)
X_train_num_normal_full = X_train_num[y_bin == 0] #Only training on normal samples for RCF
rcf_model.fit_predict(X_train_num_normal_full)

rcf_model.save_model(r"Saves/rcf_base.pkl")

Building RRCF Forest (40 trees)...


RRCF Training & Scoring: 100%|██████████| 37000/37000 [06:09<00:00, 100.18it/s]


RCF state successfully saved to Saves/rcf_base.pkl


In [6]:
from sklearn.metrics import confusion_matrix, classification_report
from catboost import CatBoostClassifier

print("\n=======================================================")
print("FINAL HOLD-OUT TEST SET EVALUATION")
print("=======================================================\n")

test_file_path = r"UNSW-NB15\unsw_test.csv"
X_test_cat, X_test_num, y_test_bin, cat_cols_test, y_test_multi = prepare_datasets(test_file_path, is_train=False)

# 2. Load the Frozen Base Models
print("\nLoading frozen base models from disk...")
loaded_catboost = CatBoostClassifier()
loaded_catboost.load_model("Saves/catboost_base.cbm")
loaded_rcf = RCF.load_model("Saves/rcf_base.pkl")

# 3. Generate Level-1 Base Scores
print("Generating base model scores (CatBoost & RCF)...")
cat_scores_test = loaded_catboost.predict_proba(X_test_cat)[:, 1]

# Use predict_proba for the test set to ensure Zero Data Leakage (Ghost Insertion)
rcf_scores_test_norm = loaded_rcf.predict_proba(X_test_num)

# 4. Fuse Scores using the trained Meta-Learner
print("Fusing scores through the Cost-Sensitive Meta-Learner...")
X_meta_test = np.column_stack((cat_scores_test, rcf_scores_test_norm))
loaded_meta_learner = CostSensitiveMetaLearner.load_model("Saves/meta_learner.pkl")
test_final_risk = loaded_meta_learner.predict_proba(X_meta_test)

# 5. Enforce the Zero Trust Boundary (Native 0.5 Threshold)
test_predictions = (test_final_risk > 0.5).astype(int)

# 6. Calculate Final SOC Operational Metrics
tn, fp, fn, tp = confusion_matrix(y_test_bin, test_predictions).ravel()
final_soc_cost = (fn * 10) + (fp * 2) 

print("\n--- FINAL OPERATIONAL REPORT (TEST SET) ---")
print(classification_report(y_test_bin, test_predictions, target_names=["Normal (0)", "Attack (1)"]))

print("\n--- ZERO TRUST SOC IMPACT ---")
print(f"True Negatives (Traffic Safely Allowed): {tn}")
print(f"True Positives (Attacks Successfully Blocked): {tp}")
print(f"False Positives (Unjustified Alert Fatigue): {fp}")
print(f"False Negatives (Missed Attacks): {fn}")
print("-------------------------------------------------------")
print(f"Total Operational Penalty Score: {final_soc_cost}")


FINAL HOLD-OUT TEST SET EVALUATION

Loading and transforming data from UNSW-NB15\unsw_test.csv (Mode: Test)...
Final Categorical features: 6
Numerical features processed: 20 Principal Components

Loading frozen base models from disk...
RCF state successfully loaded from Saves/rcf_base.pkl
Generating base model scores (CatBoost & RCF)...


RRCF Blind Test Scoring: 100%|██████████| 175341/175341 [24:14<00:00, 120.54it/s]


Fusing scores through the Cost-Sensitive Meta-Learner...
Model state successfully loaded from Saves/meta_learner.pkl

--- FINAL OPERATIONAL REPORT (TEST SET) ---
              precision    recall  f1-score   support

  Normal (0)       0.99      0.80      0.88     56000
  Attack (1)       0.91      1.00      0.95    119341

    accuracy                           0.93    175341
   macro avg       0.95      0.90      0.92    175341
weighted avg       0.94      0.93      0.93    175341


--- ZERO TRUST SOC IMPACT ---
True Negatives (Traffic Safely Allowed): 44562
True Positives (Attacks Successfully Blocked): 118959
False Positives (Unjustified Alert Fatigue): 11438
False Negatives (Missed Attacks): 382
-------------------------------------------------------
Total Operational Penalty Score: 26696
